# 03 Feature Engineering

This notebook focuses on the feature engineering pipeline for the Sentinel Sepsis-Associated AKI (SA-AKI) Early Warning System.

## Strategy
- **Missingness Indicators**: Missing data in EHR is highly informative. We explicitly create missingness flags before imputation.
- **Rolling Windows**: To capture trends in vital signs and lab results, we compute rolling aggregates (mean, max, min, std) over **6h and 12h** windows.
- **Creatinine Velocity**: The rate of change in serum creatinine is a critical physiological marker for acute kidney injury.
- **Composite Features**: Combining related measurements (e.g., BUN-to-creatinine ratio, anion-gap proxy, MAP × creatinine-velocity interaction).

In [ ]:
# Colab setup — mount Drive and cd into the project so ../data/... paths resolve.
# Safe anywhere: does nothing when not running on Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os, sys
    PROJ = '/content/drive/MyDrive/sentinel-poc/project_sentinel'
    os.chdir(os.path.join(PROJ, 'notebooks'))
    sys.path.insert(0, PROJ)
except ModuleNotFoundError:
    pass  # not on Colab — assume already running from notebooks/
import os
print('cwd =', os.getcwd())

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys, os
sys.path.append(os.path.abspath('..'))

from src import features

interim_dir = Path('../data/interim')

print("Loading defined cohort...")
cohort_df = pd.read_parquet(interim_dir / 'cohort_defined.parquet')
print(f"Loaded {cohort_df['patient_id'].nunique()} stays, {len(cohort_df):,} rows.")

print("Loading baseline creatinine...")
baseline_cr = pd.read_parquet(interim_dir / 'baseline_creatinine.parquet')
print(f"Loaded {len(baseline_cr)} baselines.")

In [ ]:
print("Adding missingness indicators...")
df_with_missingness = features.add_missingness_indicators(cohort_df)
print(f"Dataset shape after missingness indicators: {df_with_missingness.shape}")

In [ ]:
print("Adding rolling features...")
df_rolling = features.add_rolling_features(df_with_missingness)
print(f"Dataset shape after rolling features: {df_rolling.shape}")

print("Imputing missing values...")
df_imputed = features.impute_values(df_rolling)
print(f"Dataset shape after imputation: {df_imputed.shape}")

## Creatinine Velocity

Creatinine velocity is perhaps the most important feature for predicting SA-AKI. It captures the **rate of change** of serum creatinine over a recent **12-hour** window.

A rapid rise in creatinine is a stronger predictor of imminent kidney injury than the absolute level of creatinine alone, as it directly reflects acute loss of glomerular filtration.

In [ ]:
print("Calculating creatinine velocity...")
df_cr_velocity = features.add_creatinine_velocity(df_imputed, baseline_cr)
print(f"Dataset shape after creatinine velocity: {df_cr_velocity.shape}")

In [ ]:
print("Adding composite features...")
df_composites = features.add_composite_features(df_cr_velocity)
print(f"Dataset shape after composite features: {df_composites.shape}")

In [ ]:
print("Running the full feature-engineering pipeline...")
# Cells 3–7 above illustrate each step; engineer_features runs them in the
# correct order (missingness → rolling on sparse data → impute → velocity → composites).
final_features_df = features.engineer_features(cohort_df, baseline_cr)
print(f"Final dataset shape: {final_features_df.shape}")

features_path = interim_dir / 'engineered_features.parquet'
final_features_df.to_parquet(features_path, index=False)
print(f"Saved → {features_path}")

In [ ]:
print("Creating feature manifest...")
manifest = features.create_feature_manifest(
    final_features_df, interim_dir / 'feature_manifest.csv'
)
print(f"Generated manifest with {len(manifest)} features → {interim_dir / 'feature_manifest.csv'}")
display(manifest.head(15))

## Next Step

Engineered features are saved to `data/interim/engineered_features.parquet` (raw clinical columns are retained alongside the derived features, so labeling can still read `Creatinine`, `urine_rate`, etc.).

The chronological **train/val/test split happens in `04_label_engineering.ipynb`** — *after* the KDIGO labels are attached — so each split carries features **and** labels together with no leakage.